In [2]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
import numpy as np
import time
from IPython.display import clear_output

class ADXL362:
    G_TO_MS2 = 9.81
    
    # Register addresses
    RESET     = 0x1F
    FILTER    = 0x2C
    POWER_CTL = 0x2D
    
    def __init__(self, base, pmoda_lib):
        try:
            self.device = pmoda_lib.spi_open(3, 2, 1, 0)  # SCLK, MISO, MOSI, CS
            if not self.device:
                raise RuntimeError("Failed to initialize SPI device")
            self.device.configure(0, 0)  # Mode 0
            time.sleep(0.1)
            self.initialized = False
        except:
            self.device = None
            print("ADXL362 initialization failed")
            
    def read_reg(self, addr):
        if not self.device:
            return 0
        try:
            cmd = np.array([0x0B, addr, 0], dtype=np.uint8)
            data = np.zeros(3, dtype=np.uint8)
            self.device.transfer(cmd, data, 3)
            return int(data[2])
        except:
            return 0
        
    def write_reg(self, addr, value):
        if not self.device:
            return
        try:
            value = value & 0xFF
            cmd = np.array([0x0A, addr, value], dtype=np.uint8)
            data = np.zeros(3, dtype=np.uint8)
            self.device.transfer(cmd, data, 3)
            time.sleep(0.001)
        except:
            pass
        
    def init_sensor(self):
        if not self.device:
            return False
            
        try:
            # Soft reset
            self.write_reg(self.RESET, 0x52)
            time.sleep(0.1)
            
            # Verify IDs
            devid = self.read_reg(0x00)
            mstid = self.read_reg(0x01)
            partid = self.read_reg(0x02)
            
            if devid != 0xAD or mstid != 0x1D or partid != 0xF2:
                return False
                
            # Configure sensor
            self.write_reg(self.POWER_CTL, 0x00)  # Standby
            time.sleep(0.1)
            self.write_reg(self.FILTER, 0x13)     # ±2g, 100Hz
            self.write_reg(self.POWER_CTL, 0x02)  # Measurement mode
            time.sleep(0.1)
            
            self.initialized = True
            return True
        except:
            return False
        
    def read_xyz(self):
        if not self.device or not self.initialized:
            return (0, 0, 0)
            
        try:
            # Read all axis data
            xl = self.read_reg(0x0E)
            xh = self.read_reg(0x0F)
            yl = self.read_reg(0x10)
            yh = self.read_reg(0x11)
            zl = self.read_reg(0x12)
            zh = self.read_reg(0x13)
            
            # Combine and convert to signed values
            x = ((xh & 0x0F) << 8) | xl
            y = ((yh & 0x0F) << 8) | yl
            z = ((zh & 0x0F) << 8) | zl
            
            x = np.int16(x if x < 0x800 else -(0x1000 - x))
            y = np.int16(y if y < 0x800 else -(0x1000 - y))
            z = np.int16(z if z < 0x800 else -(0x1000 - z))
            
            # Convert to m/s²
            scale = 1000.0
            return (x/scale) * self.G_TO_MS2, (y/scale) * self.G_TO_MS2, (z/scale) * self.G_TO_MS2
        except:
            return (0, 0, 0)

class MAXSONAR:
    def __init__(self, base, arduino_lib):
        try:
            self.uart = arduino_lib.uart_open(1, 0)  # TXD=1 (unused), RXD=0
            self.initialized = True
        except:
            self.uart = None
            self.initialized = False
            print("MAXSONAR initialization failed")
        
    def read_distance(self):
        """Read a complete distance packet from the sensor."""
        if not self.uart or not self.initialized:
            return -1
            
        try:
            timeout = time.time() + 0.5  # 500ms timeout
            
            while time.time() < timeout:
                # Wait for 'R'
                byte = [0x00]
                self.uart.read(byte, 1)
                if chr(byte[0]) != 'R':
                    continue
                    
                # Read 3 digits and CR
                data = []
                for _ in range(4):
                    byte = [0x00]
                    self.uart.read(byte, 1)
                    data.append(byte[0])
                    
                try:
                    distance_str = ''.join(chr(b) for b in data[:3])
                    if data[3] == 0x0d and distance_str.isdigit():
                        return int(distance_str)
                except:
                    continue
                    
            return -1  # Timeout
        except:
            return -1

def make_bar(value, center=40, scale=8/9.81):
    length = int(abs(value) * scale)
    length = min(length, center - 1)
    if value > 0:
        return " " * center + "#" * length + " {:7.3f}".format(value)
    else:
        return " " * (center - length) + "#" * length + " " * (center - (center - length)) + " {:7.3f}".format(value)

def display_readings(accel_data, distance):
    clear_output(wait=True)
    x, y, z = accel_data
    
    output = []
    output.append("Unified Sensor Readings")
    output.append("=" * 50)
    
    # Accelerometer section
    output.append("\nADXL362 Accelerometer")
    output.append("-" * 25)
    if x == y == z == 0:
        output.append("Status: FAIL - Check connections")
    else:
        output.append("X-axis: {:7.3f} m/s²".format(x))
        output.append("Y-axis: {:7.3f} m/s²".format(y))
        output.append("Z-axis: {:7.3f} m/s²".format(z))
        
        output.append("\nAcceleration Visualization:")
        output.append("X: " + make_bar(x))
        output.append("Y: " + make_bar(y))
        output.append("Z: " + make_bar(z))
    
    # Distance section
    output.append("\nMAXSONAR Distance Sensor")
    output.append("-" * 25)
    if distance == -1:
        output.append("Status: FAIL - Check connections")
    else:
        output.append(f"Distance: {distance} inches ({distance * 2.54:.1f} cm)")
        if distance == 255:
            output.append("Status: No object detected/Maximum range")
        elif distance == 6:
            output.append("Status: Object at minimum range")
        else:
            output.append("Status: Valid intermediate reading")
            
        # Distance visualization
        bar_length = min(int(distance/4), 40)
        output.append("\nDistance Visualization (each # = ~4 inches):")
        output.append("0" + "-" * 39 + "160 inches")
        output.append("#" * bar_length)
    
    output.append("\nPress Ctrl+C to exit")
    print('\n'.join(output))

def main():
    print("Initializing sensors...")
    base = BaseOverlay('base.bit')
    
    # Initialize PMODA for accelerometer
    pmoda_lib = MicroblazeLibrary(base.PMODA, ['spi'])
    accel = ADXL362(base, pmoda_lib)
    accel.init_sensor()  # Try to initialize but continue even if it fails
    
    # Initialize Arduino pins for MAXSONAR
    arduino_lib = MicroblazeLibrary(base.iop_arduino, ['uart'])
    sonar = MAXSONAR(base, arduino_lib)
    
    print("Starting unified sensor display...")
    time.sleep(1)
    
    try:
        while True:
            # Read both sensors
            accel_data = accel.read_xyz()
            distance = sonar.read_distance()
            
            # Update display
            display_readings(accel_data, distance)
            time.sleep(0.05)  # 20Hz update rate
            
    except KeyboardInterrupt:
        print("\nExiting...")
    finally:
        if hasattr(accel, 'device') and accel.device:
            accel.device.close()
        if hasattr(sonar, 'uart') and sonar.uart:
            sonar.uart.close()
        print("Sensors disconnected")

if __name__ == "__main__":
    main()

Unified Sensor Readings

ADXL362 Accelerometer
-------------------------
X-axis:  -0.304 m/s²
Y-axis:   1.177 m/s²
Z-axis:  11.625 m/s²

Acceleration Visualization:
X:                                           -0.304
Y:                                            1.177
Z:                                         #########  11.625

MAXSONAR Distance Sensor
-------------------------
Distance: 9 inches (22.9 cm)
Status: Valid intermediate reading

Distance Visualization (each # = ~4 inches):
0---------------------------------------160 inches
##

Press Ctrl+C to exit

Exiting...
Sensors disconnected
